# IterGen after the three fixes, on the 13 tasks it lost in the report

**Runtime: L4 GPU is enough and the cheapest** (any GPU except T4 works; the output folder records
which one). Then **Runtime > Run all**. Nothing to edit. About 15-20 minutes, most of it setup.

Code commit 1c9a4c5 has all three fixes:
- ace68e3: `_is_valid` encodes the remainder to bytes, so unfinished strings are no longer rejected (class A)
- fa1fa66: an unfinished regex that the lexer reads as division is accepted (class B1)
- 1c9a4c5: the harness no longer counts braces inside strings when deciding the function is closed (Check 2)

This runs IterGen (checks off) once on the tasks unconstrained decoding passes and IterGen failed in
the report: 8 for Qwen2.5-3B-Instruct, 5 for Qwen3.5-4B. No before/after comparison: the "report"
column is the report's G4 run. Output on Drive: `final/<model>/itergen_off_after_fixes_<GPU>_1c9a4c5/`.
The runtime disconnects itself at the end.

In [ ]:
REPO_URL = 'https://github.com/BowenX307/constrained-llm-js-codegen-final.git'
COMMIT   = '1c9a4c53e568f5f8f16209c204cfbe8c413ced03'   # ace68e3 + fa1fa66 + 1c9a4c5
MODE     = 'itergen_off'
# task -> its class in the report: A unfinished string, B1 unfinished regex starting with a letter,
# B2 regex with a space (the grammar cannot express it), C no masks at all (not a grammar problem)
TASKS = {
    'Qwen/Qwen2.5-3B-Instruct': {'mbpp_230_replace_blank': 'A', 'mbpp_628_replace_spaces': 'A',
                                 'mbpp_757_count_reverse_pairs': 'A', 'mbpp_771_check_expression': 'A',
                                 'mbpp_772_remove_length': 'A', 'mbpp_285_text_match_two_three': 'B1',
                                 'mbpp_787_text_match_three': 'B1', 'mbpp_604_reverse_words': 'C'},
    'Qwen/Qwen3.5-4B':          {'mbpp_565_split': 'A', 'mbpp_757_count_reverse_pairs': 'A',
                                 'mbpp_619_move_num': 'A', 'mbpp_285_text_match_two_three': 'B1',
                                 'mbpp_230_replace_blank': 'B2'},
}
REPORT_TAG = '2c14eef'          # the report's runs (G4)
AUTO_DISCONNECT = True

In [ ]:
import os, sys, shutil, subprocess, time, json
CACHE = '/content/cache'
DRIVE_ROOT = '/content/drive/MyDrive/constrained-llm-js-codegen'
MS_DRIVE = f'{DRIVE_ROOT}/mask_store_zst'
os.makedirs(CACHE, exist_ok=True)
smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = smi.stdout.strip().splitlines()[0] if smi.returncode == 0 and smi.stdout.strip() else ''
print('GPU', GPU_NAME or 'none')
if not GPU_NAME or 'T4' in GPU_NAME:
    raise RuntimeError(f'This runtime has {GPU_NAME or "no GPU"}; pick L4 (Runtime > Change runtime type).')
GPU = next((short for key, short in [('RTX PRO 6000', 'G4'), ('A100', 'A100'), ('H100', 'H100'), ('L4', 'L4')]
            if key in GPU_NAME), GPU_NAME.replace(' ', '_'))
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/code'
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
for cmd in (['git', 'init', '-q', REPO], ['git', '-C', REPO, 'remote', 'add', 'origin', REPO_URL],
            ['git', '-C', REPO, 'fetch', '-q', '--depth', '1', 'origin', COMMIT],
            ['git', '-C', REPO, 'checkout', '-q', 'FETCH_HEAD']):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + ' failed:\n' + r.stderr)
def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(a) + '\n' + r.stderr[-2000:])
from importlib.metadata import version
if version('transformers') != '5.20.1':
    was = version('transformers')
    pip('transformers==5.20.1')
    if 'transformers' in sys.modules:
        raise RuntimeError(f'transformers was {was}; installed 5.20.1. Runtime > Restart session, then Run all.')
pip('lark', 'regex', 'networkx', 'fire', 'interegular', 'tqdm', 'datasets', 'jsonschema', 'psutil')
if subprocess.run([sys.executable, '-c', 'from syncode.parsers.itergen_parser import IGParser'],
                  cwd=REPO, capture_output=True).returncode != 0:
    pip('--no-deps', '--ignore-requires-python', '--force-reinstall',
        'git+https://github.com/structuredllm/syncode.git@4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4')
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], capture_output=True)
ENV = dict(os.environ, SYNCODE_CACHE=CACHE + '/syncode/', HF_HOME=CACHE + '/hf_home', PYTHONPATH=REPO,
           PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
print('ready: code', COMMIT[:7], '| GPU', GPU, '| node', shutil.which('node') is not None)

In [ ]:
def restore_mask_store(model):
    grammar = f'{REPO}/generators/grammars/javascript.lark'
    probe = subprocess.run([sys.executable, '-c',
        'import syncode.common as c; from syncode.parsers.grammars import Grammar\n'
        f'tok = c.load_tokenizer({model!r}); g = Grammar({grammar!r})\n'
        'print("MS_PATH=" + c.SYNCODE_CACHE + "mask_stores/" + type(tok).__name__ + '
        '"/grammar_strict_" + g.hash() + "_" + str(tok.vocab_size) + ".pkl")'],
        env=ENV, cwd=REPO, capture_output=True, text=True)
    lines = [l for l in probe.stdout.splitlines() if l.startswith('MS_PATH=')]
    if probe.returncode != 0 or not lines:
        raise RuntimeError('could not resolve the mask store path:\n' + probe.stderr[-2000:])
    path = lines[-1][len('MS_PATH='):]
    zst = f'{MS_DRIVE}/{os.path.basename(path)}.zst'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not (os.path.exists(path) and os.path.getsize(path) > 100 * 2**20):
        if not os.path.exists(zst):
            raise RuntimeError(f'mask store not on Drive: {zst}')
        r = subprocess.run(['zstd', '-q', '-d', '-f', '-T0', zst, '-o', path + '.part'], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('the copy on Drive is unreadable: ' + r.stderr[-300:])
        os.replace(path + '.part', path)
    print('mask store ready:', os.path.basename(path))

def status_of(run_dir):
    p = f'{run_dir}/eval.jsonl'
    return {json.loads(l)['name']: json.loads(l)['status'] for l in open(p)} if os.path.exists(p) else {}

quiet = ('Warning', 'warn(', 'Loading weights', 'torch_dtype', 'httpx')
try:
    for model, tasks in TASKS.items():
        slug = model.split('/')[-1]
        print(f'\n######## {slug}')
        restore_mask_store(model)
        tasks_file = f'/content/work/{slug}_tasks.txt'
        os.makedirs('/content/work', exist_ok=True)
        open(tasks_file, 'w').write('\n'.join(tasks) + '\n')
        name = f'{MODE}_after_fixes_{GPU}_{COMMIT[:7]}'
        out = f'/content/work/{slug}/{name}'
        r = subprocess.run([sys.executable, 'run.py', '--model', model, '--mode', MODE, '--out', out,
                            '--tasks_file', tasks_file], cwd=REPO, env=ENV,
                           text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print('\n'.join(l for l in r.stdout.splitlines() if not any(s in l for s in quiet))[-2000:])
        if r.returncode != 0:
            raise RuntimeError(f'{slug}: run.py failed')
        if shutil.which('node'):
            subprocess.run([sys.executable, 'evaluate.py', out], cwd=REPO, capture_output=True, text=True)
        shutil.copytree(out, f'{DRIVE_ROOT}/final/{slug}/{name}', dirs_exist_ok=True)
        unc = status_of(f'{DRIVE_ROOT}/final/{slug}/unconstrained_{REPORT_TAG}')
        rep = status_of(f'{DRIVE_ROOT}/final/{slug}/{MODE}_{REPORT_TAG}')
        now = status_of(out)
        print(f'\n== {slug}  (saved to Drive final/{slug}/{name})')
        print(f'{"task":32} {"class":6} {"unconstr.":12} {"report":16} {"after fixes":16} {"masks":6} {"fallback":9} stop')
        for n, cls in tasks.items():
            s = json.load(open(f'{out}/{n}.json'))['stats']
            print(f'{n:32} {cls:6} {unc.get(n, "?"):12} {rep.get(n, "?"):16} {now.get(n, "(no node)"):16} '
                  f'{s.get("mask_applied", "?")!s:6} {s.get("fallback_used", "?")!s:9} {s.get("stop_reason")}')
        for n in tasks:
            d = json.load(open(f'{out}/{n}.json'))
            print(f'\n--- {n}  ({now.get(n, "?")})')
            print(d['completion'].rstrip())
    print('\nALL DONE')
finally:
    if AUTO_DISCONNECT:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        time.sleep(30)
        runtime.unassign()